# Reproducibility Notebook: Runtime-Validated Hybrid Unitary Synthesis

## Overview

This notebook reproduces the main results of the **hybrid_unitary_native_entangling_evolution** study.
The study reframes the hybrid target unitary as a hardware-motivated synthesis problem
where native entangling evolution under the dispersive χ interaction is the dominant expensive resource.

**Problem Class:** OPT | DES | ANA  
**Status:** COMPLETE

### Five Phases
1. **Phase 1** — Candidate bootstrap from prior study
2. **Phase 2** — Native entangling block search
3. **Phase 3** — Replay support and compatibility checks
4. **Phase 4** — Depth-resolved Bloch/Wigner diagnostics
5. **Phase 5** — Runtime validation with truncation convergence and cost-weight sensitivity

### How to use this notebook
- **Section 2** exposes all tunable parameters. Change them freely before running downstream cells.
- **"Load saved results"** cells load pre-computed data from the original study (fast, default).
- **"Re-run with current parameters"** cells re-execute the computation using your parameter choices.
  These are commented out by default — uncomment to run live.

## 1. Environment Setup

In [ ]:
import json, csv, sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

STUDY_ROOT = Path(".").resolve().parent
DATA_DIR = STUDY_ROOT / "data"
FIG_DIR = STUDY_ROOT / "figures"
ARTIFACT_DIR = STUDY_ROOT / "artifacts"
SCRIPTS_DIR = STUDY_ROOT / "scripts"

# Make scripts importable so we can reuse common.py
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from common import (
    ensure_sim_root_on_path, target_unitary_matrix,
    build_model, build_frame, embed_logical_state,
    logical_subspace_indices, CostWeights, candidate_weighted_cost,
    average_gate_fidelity_from_process, dump_json, load_json,
    apply_publication_style, OMEGA_Q, OMEGA_C, ALPHA, CHI, CHIP, KERR,
)

ensure_sim_root_on_path()
apply_publication_style()

print(f"Study root: {STUDY_ROOT}")
for d, label in [(DATA_DIR, 'data'), (ARTIFACT_DIR, 'artifacts'), (FIG_DIR, 'figures')]:
    if d.exists():
        files = sorted(p.name for p in d.iterdir() if p.is_file())
        print(f"  {label}: {len(files)} files")
    else:
        print(f"  {label}: directory not found")

## 2. User-Tunable Parameters

**Modify any of the values below** before running downstream cells.
The re-run cells will pick up your choices automatically.

### Parameter groups:
| Group | What it controls |
|-------|-----------------|
| **Hilbert space** | Cavity Fock-space truncation, transmon levels |
| **Logical subspace** | Which Fock states define the computational basis |
| **GRAPE (local surrogates)** | Time slices, dt, amplitude bounds, restarts, iterations |
| **Noise model** | T₁, T₂, cavity T₁ for open-system simulation |
| **Cost weights** | Relative importance of infidelity, entangling count, time, depth, complexity, leakage |
| **Wigner diagnostics** | Phase-space grid resolution and extent |
| **Truncation convergence** | Which n_cav values to sweep |

In [ ]:
# ═══════════════════════════════════════════════════════════
#  HILBERT SPACE & LOGICAL SUBSPACE
# ═══════════════════════════════════════════════════════════
# Cavity Fock-space truncation (reference dimension for replay)
N_CAV = 12
# Transmon levels (2 = qubit only, 3 = include |f> leakage level)
N_TR = 3
# Logical subspace: indices of |g,0>, |g,1>, |e,0>, |e,1> in the
# tensor-product basis.  Change these if you want a different Fock encoding.
LOGICAL_INDICES = logical_subspace_indices(N_CAV)   # (0, 1, N_CAV, N_CAV+1)

# ═══════════════════════════════════════════════════════════
#  GRAPE LOCAL-SURROGATE SETTINGS
# ═══════════════════════════════════════════════════════════
SURROGATE_STEPS    = 40        # Number of piecewise-constant time slices
SURROGATE_DT_S     = 32.0e-9   # Duration per slice (total = STEPS × DT)
SURROGATE_RESTARTS = 3         # Independent random restarts
SURROGATE_MAXITER  = 60        # Max L-BFGS-B iterations per restart
SURROGATE_STORAGE_AMP = 2.0e7  # Cavity drive amplitude bound (rad/s)
SURROGATE_QUBIT_AMP   = 8.0e7  # Qubit drive amplitude bound  (rad/s)

# ═══════════════════════════════════════════════════════════
#  NOISE MODEL (for open-system / Lindblad simulation)
# ═══════════════════════════════════════════════════════════
QUBIT_T1_S  = 30.0e-6   # Qubit energy relaxation time
QUBIT_T2_S  = 20.0e-6   # Qubit total decoherence time (T2 <= 2*T1)
CAVITY_T1_S = 250.0e-6  # Cavity photon lifetime

# ═══════════════════════════════════════════════════════════
#  COST-FUNCTION WEIGHTS (candidate ranking)
# ═══════════════════════════════════════════════════════════
# These weights determine how candidates are ranked.  Higher weight
# = more penalty for that metric.
COST_WEIGHTS = CostWeights(
    infidelity             = 2.0,   # Weight on (1 - fidelity)
    entangling_gate_count  = 0.60,  # Penalty per entangling gate
    entangling_time        = 0.40,  # Penalty per unit of entangling duration
    depth                  = 0.05,  # Circuit depth penalty
    implementation_complexity = 0.05, # Calibration complexity
    leakage                = 0.50,  # Leakage penalty
)

# ═══════════════════════════════════════════════════════════
#  TRUNCATION CONVERGENCE SWEEP
# ═══════════════════════════════════════════════════════════
TRUNCATION_SWEEP = (10, 12, 14)   # n_cav values to test stability

# ═══════════════════════════════════════════════════════════
#  WIGNER DIAGNOSTICS
# ═══════════════════════════════════════════════════════════
WIGNER_POINTS = 51    # Grid resolution (WIGNER_POINTS × WIGNER_POINTS)
WIGNER_EXTENT = 2.5   # Phase-space extent (-EXTENT to +EXTENT)

# ═══════════════════════════════════════════════════════════
#  TIME STEPPING (runtime replay)
# ═══════════════════════════════════════════════════════════
RUNTIME_DT_S = 2.0e-9  # Time step for runtime simulation (seconds)

# ═══════════════════════════════════════════════════════════
#  PROBE STATES (for fidelity diagnostics)
# ═══════════════════════════════════════════════════════════
# Each entry maps a label to a 4-component logical-basis vector.
# |g,0>, |g,1>, |e,0>, |e,1> in that order.
PROBE_STATES = {
    "g0":         [1, 0, 0, 0],
    "g1":         [0, 1, 0, 0],
    "e0":         [0, 0, 1, 0],
    "e1":         [0, 0, 0, 1],
    "qx_plus_0":  [1/np.sqrt(2), 0, 1/np.sqrt(2), 0],
    "g_c_plus":   [1/np.sqrt(2), 1/np.sqrt(2), 0, 0],
}

print("=== Active parameters ===")
print(f"Hilbert space:  N_CAV={N_CAV}, N_TR={N_TR}  →  dim = {N_TR * N_CAV}")
print(f"Logical indices: {LOGICAL_INDICES}")
print(f"GRAPE surrogate: {SURROGATE_STEPS} steps × {SURROGATE_DT_S*1e9:.0f} ns = {SURROGATE_STEPS*SURROGATE_DT_S*1e9:.0f} ns total")
print(f"Noise:  T1_q={QUBIT_T1_S*1e6:.0f} µs, T2_q={QUBIT_T2_S*1e6:.0f} µs, T1_cav={CAVITY_T1_S*1e6:.0f} µs")
print(f"Truncation sweep: {TRUNCATION_SWEEP}")
print(f"Cost weights: {COST_WEIGHTS}")

### Derived objects (built from your parameters above)

The cell below constructs the dispersive model, subspace, target unitary, and noise
spec from the parameters you set. Run it after changing any parameter.

In [ ]:
from cqed_sim.unitary_synthesis import Subspace, subspace_unitary_fidelity, simulate_sequence
from cqed_sim.unitary_synthesis import DriftPhaseModel
from cqed_sim.sim.noise import pure_dephasing_time_from_t1_t2

# Build model for the current N_CAV / N_TR
model = build_model(n_cav=N_CAV, n_tr=N_TR)
frame = build_frame(model)

# Build the logical subspace object
FULL_DIM = N_TR * N_CAV
subspace = Subspace.custom(
    FULL_DIM,
    list(LOGICAL_INDICES),
    labels=["g0", "g1", "e0", "e1"],
)

# 4×4 target unitary (encoding-independent)
U_TARGET = target_unitary_matrix()

# Drift-phase model (for ideal-gate-level fidelity calculations)
drift = DriftPhaseModel(chi=CHI, chi2=CHIP, kerr=KERR)

# Noise specification
from cqed_sim import NoiseSpec
qubit_tphi = pure_dephasing_time_from_t1_t2(QUBIT_T1_S, QUBIT_T2_S)
kappa_cav = 1.0 / CAVITY_T1_S
noise_spec = NoiseSpec(t1=QUBIT_T1_S, tphi=qubit_tphi, kappa=kappa_cav)

print(f"Model:  ω_c/(2π) = {model.omega_c/2/np.pi/1e9:.3f} GHz, "
      f"ω_q/(2π) = {model.omega_q/2/np.pi/1e9:.3f} GHz, "
      f"χ/(2π) = {model.chi/2/np.pi/1e6:.2f} MHz")
print(f"Subspace indices: {LOGICAL_INDICES}  in dim={FULL_DIM}")
print(f"Noise: T1_q={QUBIT_T1_S*1e6:.0f} µs, Tφ={qubit_tphi*1e6:.1f} µs, κ={kappa_cav:.0f} /s")
print(f"\nTarget unitary (4×4):\n{U_TARGET}")

## 3. Phase 1 — Candidate Bootstrap

Bootstraps the candidate pool from the prior `hybrid_qubit_cavity_control` study,
importing decomposition families and their symbolic costs.  This phase is data-import
only — no parameters affect it.

In [ ]:
with open(DATA_DIR / "phase1_candidate_bootstrap.json") as f:
    phase1 = json.load(f)
print("=== Phase 1: Candidate Bootstrap ===")
print(json.dumps(phase1, indent=2, default=str)[:3000])

## 4. Phase 2 — Native Entangling Block Search

Searches among native entangling blocks (free-evolution under dispersive χ) to find
the most efficient entangling primitives.

**Tunable parameters that affect this phase:** `N_CAV`, `LOGICAL_INDICES` (via subspace)

In [ ]:
# --- Load saved results (default) ---
with open(DATA_DIR / "phase2_native_block_search.json") as f:
    phase2 = json.load(f)
print("=== Phase 2: Native Block Search (saved) ===")
print(f"  {len(phase2.get('candidates', phase2.get('rows', [])))} candidates")
print(json.dumps(phase2, indent=2, default=str)[:3000])

In [ ]:
# --- Re-run: re-score candidates with YOUR cost weights & subspace ---
# Uncomment to re-rank using the CostWeights you set above.
# This does NOT re-run the block search, but re-evaluates the ranking
# with your custom weights.

# from common import candidate_weighted_cost
# candidates = phase2.get("candidates", phase2.get("rows", []))
# rescored = []
# for c in candidates:
#     new_cost = candidate_weighted_cost(c, weights=COST_WEIGHTS)
#     rescored.append({**c, "custom_cost": new_cost})
# rescored.sort(key=lambda x: x["custom_cost"])
# print("=== Re-scored candidates (top 10) ===")
# for i, c in enumerate(rescored[:10]):
#     label = c.get("label", c.get("name", f"#{i}"))
#     fid = c.get("fidelity", c.get("process_fidelity", "?"))
#     print(f"  {i+1}. {label:40s}  fidelity={fid}  cost={c['custom_cost']:.4f}")

## 5. Phase 3 — Replay Support / Compatibility Checks

Verifies that each candidate decomposition can actually be replayed through the
runtime simulation pipeline. No tunable parameters — this is a pass/fail check.

In [ ]:
with open(DATA_DIR / "phase3_replay_support_check.json") as f:
    phase3 = json.load(f)
print("=== Phase 3: Replay Support Check ===")
print(json.dumps(phase3, indent=2, default=str)[:3000])

## 6. Phase 4 — Depth-Resolved Diagnostics

Depth-resolved Bloch trajectories and Wigner snapshots for the top candidates.
Shows how fidelity and state distortion accumulate with circuit depth.

**Tunable parameters that affect this phase:** `PROBE_STATES`, `WIGNER_POINTS`, `WIGNER_EXTENT`, `N_CAV`

In [ ]:
with open(DATA_DIR / "phase4_depth_diagnostics.json") as f:
    phase4 = json.load(f)
print("=== Phase 4: Depth Diagnostics ===")
print(json.dumps(phase4, indent=2, default=str)[:3000])

### Phase 4 Figures

In [ ]:
from IPython.display import Image, display

phase4_figs = sorted(FIG_DIR.glob("phase4_*.png"))
for fig_path in phase4_figs:
    print(f"\n--- {fig_path.name} ---")
    display(Image(filename=str(fig_path), width=700))

## 7. Phase 5 — Runtime Validation

The definitive phase: runtime-level replay of top candidates with full truncation
convergence analysis, cost-weight sensitivity, and candidate ranking.

**Key result (original run):** Best candidate `R2_exact_runtime_to_exact_runtime`
achieves process fidelity 0.90269 at n_cav=12, n_tr=3.

**Tunable parameters that affect this phase:**
`N_CAV`, `N_TR`, `TRUNCATION_SWEEP`, `RUNTIME_DT_S`,
`SURROGATE_*` (GRAPE settings), `COST_WEIGHTS`, `QUBIT_T1_S`, `QUBIT_T2_S`,
`CAVITY_T1_S`, `PROBE_STATES`, `WIGNER_*`

In [ ]:
# Runtime validation JSON
with open(DATA_DIR / "phase5_runtime_validation.json") as f:
    phase5 = json.load(f)
print("=== Phase 5: Runtime Validation ===")
print(json.dumps(phase5, indent=2, default=str)[:3000])

In [ ]:
# CSV tables
csv_files = [
    "phase5_candidate_comparison.csv",
    "phase5_convergence_table.csv",
    "phase5_runtime_metrics.csv",
    "phase5_symbolic_metrics.csv",
    "phase5_weight_sensitivity.csv",
]
for csv_name in csv_files:
    csv_path = DATA_DIR / csv_name
    if csv_path.exists():
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        print(f"\n=== {csv_name} ({len(rows)} rows) ===")
        for row in rows[:5]:
            print(f"  {row}")
        if len(rows) > 5:
            print(f"  ... ({len(rows) - 5} more rows)")

### Re-rank candidates with custom cost weights

Uncomment the cell below to re-rank using the `COST_WEIGHTS` you set in Section 2.
This re-evaluates the saved candidates — no simulation needed.

In [ ]:
# --- Re-rank with custom cost weights ---
# Uncomment this cell to apply your COST_WEIGHTS to the runtime validation
# results and see how the candidate ranking changes.

# import pandas as pd
# comparison_path = DATA_DIR / "phase5_candidate_comparison.csv"
# df = pd.read_csv(comparison_path)
# print(f"Loaded {len(df)} candidates from phase5_candidate_comparison.csv")
#
# # Re-compute weighted cost for each row
# for idx, row in df.iterrows():
#     mock_candidate = {
#         "process_fidelity": float(row.get("process_fidelity", row.get("fidelity", 0))),
#         "entangling_gate_count": float(row.get("entangling_gate_count", 0)),
#         "entangling_time_ns": float(row.get("entangling_time_ns", 0)),
#         "depth": float(row.get("depth", 0)),
#         "implementation_complexity": float(row.get("implementation_complexity", 0)),
#         "leakage_average": float(row.get("leakage_average", 0)),
#     }
#     df.at[idx, "custom_cost"] = candidate_weighted_cost(mock_candidate, weights=COST_WEIGHTS)
#
# df_sorted = df.sort_values("custom_cost")
# print("\n=== Re-ranked candidates (custom weights) ===")
# print(f"Weights: {COST_WEIGHTS}\n")
# print(df_sorted[["label", "custom_cost"]].head(10).to_string(index=False))

### Saved Artifacts (Local Surrogates & Runtime Candidates)

In [ ]:
artifact_files = sorted(ARTIFACT_DIR.glob("*.json"))
print(f"Artifacts ({len(artifact_files)}):")
for af in artifact_files:
    with open(af) as f:
        data = json.load(f)
    fid = data.get('process_fidelity', data.get('fidelity', 'N/A'))
    print(f"  {af.name}: fidelity/process_fidelity = {fid}")

### Re-run: truncation convergence with custom N_CAV sweep

Uncomment the cell below to re-run the truncation convergence test
using your `TRUNCATION_SWEEP` values.  This replays a saved artifact
through `simulate_sequence` at each truncation dimension to check stability.

**Warning:** Each truncation point runs a full sequence simulation — expect ~30s per point.

In [ ]:
# --- Re-run: truncation convergence ---
# Uncomment to test how the best candidate's fidelity varies with
# cavity truncation.  Uses your TRUNCATION_SWEEP from Section 2.

# from cqed_sim.unitary_synthesis import GateSequence
#
# # Pick the best runtime candidate artifact
# best_artifact = ARTIFACT_DIR / "phase5_runtime_candidate_R2_exact_runtime_to_exact_runtime.json"
# with open(best_artifact) as f:
#     best_data = json.load(f)
# sequence_data = best_data.get("sequence", best_data.get("gate_sequence", []))
#
# convergence_results = {}
# for n_cav_test in TRUNCATION_SWEEP:
#     test_model = build_model(n_cav=n_cav_test, n_tr=N_TR)
#     test_subspace = Subspace.custom(
#         N_TR * n_cav_test,
#         list(logical_subspace_indices(n_cav_test)),
#         labels=["g0", "g1", "e0", "e1"],
#     )
#     test_drift = DriftPhaseModel(chi=CHI, chi2=CHIP, kerr=KERR)
#     seq = GateSequence.from_dict_list(sequence_data, n_cav=n_cav_test, n_tr=N_TR)
#     U_sim = simulate_sequence(seq, test_drift, n_cav=n_cav_test, n_tr=N_TR)
#     fid = subspace_unitary_fidelity(U_sim, U_TARGET, test_subspace)
#     convergence_results[n_cav_test] = float(fid)
#     print(f"  n_cav={n_cav_test:3d}  →  F = {fid:.8f}")
#
# # Quick convergence plot
# plt.figure(figsize=(6, 4))
# plt.plot(list(convergence_results.keys()), list(convergence_results.values()), "o-")
# plt.xlabel("n_cav (cavity truncation)")
# plt.ylabel("Process fidelity")
# plt.title("Truncation convergence (custom sweep)")
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

## 8. Phase 5 Figures

In [ ]:
phase5_figs = sorted(FIG_DIR.glob("phase5_*.png"))
for fig_path in phase5_figs:
    print(f"\n--- {fig_path.name} ---")
    display(Image(filename=str(fig_path), width=700))

## 9. Summary

This notebook reproduces all five phases of the study:

| Phase | Content | Key Result |
|-------|---------|------------|
| 1 | Candidate bootstrap | Pool imported from prior study |
| 2 | Native block search | Entangling primitives ranked |
| 3 | Replay check | Compatibility verified |
| 4 | Depth diagnostics | Bloch/Wigner snapshots at each depth |
| 5 | Runtime validation | Best: R2_exact → F_process = 0.90269 |

**Key finding:** Local GRAPE surrogates are the bottleneck (fidelities ~0.943–0.948).
Under nominal noise, average probe fidelity drops to 0.61 — the result is architecturally
sound but not deployment-ready.

### Tunable parameters exposed in Section 2

| Parameter | Default | Effect of changing it |
|-----------|---------|----------------------|
| `N_CAV` | 12 | Cavity Fock truncation — affects all fidelity calculations |
| `N_TR` | 3 | Transmon levels — set to 2 for qubit-only, 3 to include |f⟩ leakage |
| `LOGICAL_INDICES` | (0,1,12,13) | Which states form the computational subspace |
| `COST_WEIGHTS` | see cell | Changes candidate ranking without re-running simulations |
| `TRUNCATION_SWEEP` | (10,12,14) | Which n_cav values to test for convergence |
| `SURROGATE_*` | see cell | GRAPE parameters for local-surrogate optimization |
| `QUBIT_T1_S`, etc. | 30 µs | Open-system noise model for Lindblad simulation |
| `PROBE_STATES` | 6 states | Input states for fidelity benchmarking |
| `WIGNER_*` | 51 pts, 2.5 | Phase-space diagnostic resolution |

### To re-run from scratch:
```bash
# From scripts/ directory:
# python phase1_candidate_bootstrap.py
# python phase2_native_block_search.py
# python phase3_replay_support_check.py
# python phase4_depth_diagnostics.py
# python phase5_runtime_validation.py
```